# Document Loaders — PDF, CSV & Web Ingestion

Turn real-world files and web pages into LangChain `Document` objects — the first step in every RAG pipeline.

---

In [ ]:
!pip install -q langchain langchain-community pypdf beautifulsoup4

---

## 1 · The Document Object

**The Problem** — LLMs only understand text. Your data lives in PDFs, spreadsheets, web pages, and dozens of other formats.

**The Solution** — Every loader outputs a standardized `Document` object with two fields: `page_content` (the text) and `metadata` (source, page number, etc.).

> Think of a Document like a library index card — one side has the content, the other has where it came from. Every card uses the same format no matter the original source.

In [ ]:
from langchain_core.documents import Document

# ── Manually create a Document to understand the structure ──
doc = Document(
    page_content="Transformers use self-attention to weigh input tokens.",  # the actual text content
    metadata={"source": "paper.pdf", "page": 0}                           # provenance info — where this came from
)

# Two fields — that's all a Document is
print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")
print(f"Type: {type(doc)}")

### What Just Happened?

We created a `Document` manually — `page_content` holds the text, `metadata` holds context about the source. Every loader in LangChain produces exactly this structure. That standardization is what makes the whole pipeline (loaders → splitters → embeddings → vector stores) modular.

---

## 2 · PyPDFLoader — Loading PDFs

**The Problem** — PDFs have complex internal structures (fonts, columns, tables). Extracting clean text isn't as simple as reading a `.txt` file.

**The Solution** — `PyPDFLoader` uses the `pypdf` library to extract text page-by-page. Each page becomes its own Document with the page number in metadata.

> Think of it as a photocopy machine that separates a bound book into individual pages and stamps the page number on the back of each copy.

In [ ]:
# ── First, let's create a sample PDF to work with ──
# (In real projects you'd point this at an existing file)

try:
    from fpdf import FPDF
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "fpdf2"])
    from fpdf import FPDF

pdf = FPDF()

# Page 1 — intro
pdf.add_page()
pdf.set_font("Helvetica", size=16)
pdf.cell(text="Introduction to Transformers", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", size=12)
pdf.multi_cell(w=0, text=(
    "The Transformer architecture was introduced in 'Attention Is All You Need' (2017). "
    "It relies entirely on self-attention mechanisms, dispensing with recurrence and convolutions. "
    "This design allows for significantly more parallelization during training."
))

# Page 2 — details
pdf.add_page()
pdf.set_font("Helvetica", size=16)
pdf.cell(text="Self-Attention Mechanism", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", size=12)
pdf.multi_cell(w=0, text=(
    "Self-attention computes a weighted sum of all positions in a sequence. "
    "Each token attends to every other token, producing context-aware representations. "
    "Multi-head attention runs this process in parallel across multiple representation subspaces."
))

# Page 3 — applications
pdf.add_page()
pdf.set_font("Helvetica", size=16)
pdf.cell(text="Applications", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", size=12)
pdf.multi_cell(w=0, text=(
    "Transformers power GPT, BERT, T5, and virtually every modern LLM. "
    "Beyond NLP, they've been adapted for computer vision (ViT), protein folding (AlphaFold), "
    "and audio generation. The architecture's flexibility makes it the backbone of modern AI."
))

pdf.output("sample_transformers.pdf")
print("Created: sample_transformers.pdf (3 pages)")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# ── Load the PDF — each page becomes a separate Document ──
loader = PyPDFLoader("sample_transformers.pdf")
docs = loader.load()                             # returns list[Document]

print(f"Total documents (pages): {len(docs)}")   # 3 — one per page
print(f"Type: {type(docs[0])}\n")

# ── Inspect each page ──
for doc in docs:
    print(f"--- Page {doc.metadata['page']} ---")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:80]}...")
    print()

### What Just Happened?

- `PyPDFLoader` read the PDF and created **one Document per page**
- Each Document's `metadata` includes `source` (file path) and `page` (zero-indexed page number)
- The `page_content` contains the extracted text — ready to pass into a text splitter or directly to an LLM

> **When to use:** Text-based PDFs with straightforward layouts. For scanned PDFs or complex multi-column layouts, consider `UnstructuredPDFLoader` or `PyMuPDFLoader`.

---

## 3 · CSVLoader — Loading Spreadsheets

**The Problem** — CSV files contain structured, row-based data. You need each row as a searchable document while preserving column context.

**The Solution** — `CSVLoader` converts each row into a Document. Column names are included in `page_content` as key-value pairs.

> Think of it as tearing a spreadsheet into individual flashcards — one card per row, with column headers printed on each so the row makes sense on its own.

In [ ]:
import csv

# ── Create a sample CSV to work with ──
data = [
    ["name", "role", "expertise", "years_exp"],
    ["Alice Chen", "ML Engineer", "Transformers, NLP", "5"],
    ["Bob Kumar", "Data Scientist", "Computer Vision, PyTorch", "3"],
    ["Carol Zhang", "MLOps Engineer", "Kubernetes, MLflow", "7"],
    ["David Park", "Research Scientist", "Reinforcement Learning", "4"],
]

with open("team.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(data)

print("Created: team.csv (4 rows)")

In [ ]:
from langchain_community.document_loaders import CSVLoader

# ── Basic CSV loading — each row becomes a Document ──
loader = CSVLoader(file_path="team.csv")
docs = loader.load()

print(f"Total documents (rows): {len(docs)}\n")   # 4 — one per data row

# ── Inspect what a row looks like as a Document ──
print("=== Row 0 as Document ===")
print(f"page_content:\n{docs[0].page_content}\n")  # key: value pairs from columns
print(f"metadata: {docs[0].metadata}")              # source file + row number

In [ ]:
# ── source_column — use a specific column as the metadata source ──
loader = CSVLoader(
    file_path="team.csv",
    source_column="name"       # metadata['source'] = the person's name instead of filename
)
docs = loader.load()

# Now each doc's source is the person's name — useful for attribution in RAG
for doc in docs:
    print(f"Source: {doc.metadata['source']:20s} | Row: {doc.metadata['row']}")

In [ ]:
# ── csv_args — handle non-standard CSV formats ──
# Create a pipe-delimited file to demonstrate
with open("team_pipe.csv", "w") as f:
    f.write("name|role|expertise\n")
    f.write("Alice Chen|ML Engineer|Transformers\n")
    f.write("Bob Kumar|Data Scientist|Computer Vision\n")

loader = CSVLoader(
    file_path="team_pipe.csv",
    csv_args={
        "delimiter": "|",      # pipe-separated instead of comma
        "quotechar": '"',
    }
)
docs = loader.load()
print(docs[0].page_content)

### What Just Happened?

- `CSVLoader` created **one Document per row** — column names become keys in the `page_content` string
- `source_column` lets you override which column appears as `metadata['source']` — useful for attribution in RAG answers
- `csv_args` passes through to Python's `csv.DictReader` — handles delimiters, quote characters, custom field names

> **When to use:** Tabular data where each row is a meaningful unit — employee records, product catalogs, FAQ lists, support tickets.

---

## 4 · WebBaseLoader — Loading Web Pages

**The Problem** — Web pages contain useful content buried under navigation bars, footers, ads, and HTML tags.

**The Solution** — `WebBaseLoader` fetches a URL, uses BeautifulSoup to parse HTML, and extracts text content. You can target specific elements for cleaner output.

> Think of it as a research assistant who visits a webpage, strips away the noise, and hands you a clean printout of just the article text.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

# ── Basic — load the full page text from a URL ──
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

print(f"Documents loaded: {len(docs)}")                 # 1 Document per URL
print(f"Content length: {len(docs[0].page_content)} chars")
print(f"\nMetadata keys: {list(docs[0].metadata.keys())}")
print(f"Source: {docs[0].metadata['source']}")
print(f"\nFirst 300 chars:\n{docs[0].page_content[:300]}")

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# ── Targeted extraction — only pull content from specific HTML elements ──
# This avoids ingesting navbars, footers, sidebars, etc.
loader = WebBaseLoader(
    web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")  # only these CSS classes
        )
    ),
)
docs = loader.load()

print(f"Content length (targeted): {len(docs[0].page_content)} chars")
print(f"\nFirst 300 chars:\n{docs[0].page_content[:300]}")
# Much cleaner — no navigation, footer, or sidebar content

In [ ]:
# ── Load multiple URLs at once ──
loader = WebBaseLoader(
    web_paths=[
        "https://lilianweng.github.io/posts/2023-06-23-agent/",
        "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    ]
)
docs = loader.load()

print(f"Documents loaded: {len(docs)}")   # 2 — one per URL
for doc in docs:
    print(f"  Source: {doc.metadata['source'][:60]}... | Length: {len(doc.page_content)} chars")

### What Just Happened?

- `WebBaseLoader` fetched the HTML, parsed it with BeautifulSoup, and extracted the text into a Document
- **Basic mode** grabs everything — you may get navigation text, footers, and other noise
- **Targeted mode** with `bs_kwargs` + `SoupStrainer` lets you isolate specific HTML classes for much cleaner output
- You can pass **multiple URLs** and get one Document per page

> **When to use:** Blogs, documentation, news articles — any static page. Does **not** handle JavaScript-rendered content. For SPAs, look into `PlaywrightURLLoader` or `SeleniumURLLoader`.

---

## 5 · DirectoryLoader — Batch Loading Files

**The Problem** — You have a folder full of PDFs and need to load them all at once.

**The Solution** — `DirectoryLoader` walks a directory and loads every matching file using a specified loader class. Use `glob` patterns to filter.

> Think of it as a librarian who walks through an entire shelf, picks up every book of a certain type, and processes each one with the right reader.

In [ ]:
import os
from fpdf import FPDF

# ── Create a directory with multiple PDFs to demonstrate ──
os.makedirs("sample_docs", exist_ok=True)

topics = {
    "attention.pdf": "Attention mechanisms allow models to focus on relevant parts of the input sequence.",
    "embeddings.pdf": "Word embeddings map tokens to dense vector representations that capture semantic meaning.",
    "fine_tuning.pdf": "Fine-tuning adapts a pretrained model to a specific downstream task using task-specific data.",
}

for filename, content in topics.items():
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", size=12)
    pdf.multi_cell(w=0, text=content)
    pdf.output(f"sample_docs/{filename}")

print(f"Created {len(topics)} PDFs in sample_docs/")
print(os.listdir("sample_docs"))

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# ── Load all PDFs from a directory in one shot ──
loader = DirectoryLoader(
    path="./sample_docs/",        # folder to scan
    glob="**/*.pdf",              # match all PDFs, including subdirectories
    loader_cls=PyPDFLoader,       # use PyPDFLoader for each file found
    show_progress=True            # progress bar — useful for large directories
)
docs = loader.load()

print(f"\nTotal documents loaded: {len(docs)}\n")

for doc in docs:
    source = os.path.basename(doc.metadata['source'])  # just the filename
    print(f"  {source:25s} | Page {doc.metadata['page']} | {len(doc.page_content)} chars")

### What Just Happened?

- `DirectoryLoader` scanned `./sample_docs/` for all `.pdf` files using the glob pattern
- Each file was loaded using `PyPDFLoader` (specified via `loader_cls`)
- All pages from all files ended up in a single flat `list[Document]`
- `show_progress=True` gives you a progress bar — essential for large directories

> **When to use:** Processing document collections — research papers, legal contracts, support tickets. Swap `loader_cls` to `CSVLoader`, `TextLoader`, etc. for other file types.

---

## 6 · load() vs lazy_load() — Memory Management

**The Problem** — `.load()` pulls everything into memory at once. Hundreds of large PDFs will eat your RAM.

**The Solution** — `.lazy_load()` returns a generator that yields one Document at a time, processing files incrementally.

> `.load()` is filling your entire shopping cart before checkout. `.lazy_load()` is a conveyor belt — items arrive one at a time and you process each before the next appears.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("sample_transformers.pdf")

# ── Eager loading — everything in memory at once ──
all_docs = loader.load()                          # returns list[Document]
print(f"load() returned: {type(all_docs)} with {len(all_docs)} docs")
print(f"All pages in memory: {[d.metadata['page'] for d in all_docs]}")

In [ ]:
# ── Lazy loading — one Document at a time via generator ──
loader = PyPDFLoader("sample_transformers.pdf")

lazy_docs = loader.lazy_load()                    # returns a generator, NOT a list
print(f"lazy_load() returned: {type(lazy_docs)}")

# Process one page at a time — only one page in memory at any point
for doc in lazy_docs:
    page = doc.metadata['page']
    char_count = len(doc.page_content)
    print(f"  Processing page {page}: {char_count} chars")
    # In production: embed, index, or transform here before moving to next

In [ ]:
# ── Real-world pattern: lazy_load + processing pipeline ──
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    path="./sample_docs/",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)

# Process documents one at a time without loading the entire corpus
doc_count = 0
total_chars = 0

for doc in loader.lazy_load():
    doc_count += 1
    total_chars += len(doc.page_content)
    # In production: split → embed → upsert to vector store here

print(f"Processed {doc_count} documents ({total_chars:,} total chars)")
print("Memory usage stayed flat — only one doc in memory at a time")

### What Just Happened?

- `.load()` returns a `list` — every document loaded into memory at once
- `.lazy_load()` returns a `generator` — yields one document at a time, constant memory usage
- The processing pattern is identical (`for doc in ...`), only the memory profile changes

> **Key insight:** Default to `.load()` for prototyping and small datasets. Switch to `.lazy_load()` when working with large document collections in production.

---

## 7 · Putting It Together — Loader → Splitter Preview

Document loaders are step 1 in the RAG pipeline. Here's how they connect to the next step — text splitting (covered in Tutorial 05).

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Step 1: Load ──
loader = PyPDFLoader("sample_transformers.pdf")
docs = loader.load()
print(f"Loaded: {len(docs)} pages")

# ── Step 2: Split (preview — Tutorial 05 covers this in depth) ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,           # max characters per chunk
    chunk_overlap=30,         # overlap to preserve context across chunks
)
chunks = splitter.split_documents(docs)
print(f"Split into: {len(chunks)} chunks\n")

# ── Inspect a few chunks ──
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} (page {chunk.metadata['page']}) ---")
    print(f"{chunk.page_content}")
    print()

### What Just Happened?

This is the standard two-step pattern: **load → split**. The loader produces page-level Documents, the splitter breaks them into smaller chunks sized for embedding models. Metadata propagates through — each chunk still knows which page and file it came from.

Tutorial 05 covers text splitters in detail — recursive, token-based, and semantic chunking strategies.

---

## Key Takeaways

| Loader | Code | Returns | When to Use |
|--------|------|---------|-------------|
| PyPDFLoader | `PyPDFLoader("file.pdf")` | 1 Doc per page | Text-based PDFs |
| CSVLoader | `CSVLoader("file.csv")` | 1 Doc per row | Tabular data, FAQs |
| WebBaseLoader | `WebBaseLoader("https://...")` | 1 Doc per URL | Static web pages |
| DirectoryLoader | `DirectoryLoader("./dir/", glob="*.pdf", loader_cls=...)` | All matching files | Batch loading |
| `.load()` | `loader.load()` | `list[Document]` | Prototyping |
| `.lazy_load()` | `loader.lazy_load()` | `Generator[Document]` | Production / large data |

**Next →** [05 · Text Splitters](../05-text-splitters/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*